## METHODS COMPARISON USING CIFAR100 WITH 10 CLASSES

On this file, the logistic regression proccess is done in order to compare it with the results of the Structure Predictions package. \
We find out that there are some considerable differences. 

For this exercice we use the CIFAR100 dataset provided by Keras. This is a dataset of 50,000 32x32 color training images and 10,000 test images, labeled over 100 fine-grained classes that are grouped into 20 coarse-grained classes. 

In this file we do a subset of the dataset in order to have less categories. For this case, we implement a 10 categories subset as following.

In [ ]:
from sklearn import metrics

In [ ]:
from keras.datasets import cifar100
import matplotlib.pyplot as plt
import numpy as np

import time


# Download dataset of CIFAR-100 (Canadian Institute for Advanced Research)
(X_train,Y_train),(X_test,Y_test) = cifar100.load_data()
 
# Check the shape of the array
print('x_train shape:', X_train.shape)
print('x_test shape:', X_test.shape)
print('y_train shape:', Y_train.shape)
print('y_test shape:', Y_test.shape)
 
# Number of data set samples 
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')
 
# Data format confirmation
print(type(X_test))
print(type(Y_test[0]))

### Organizing data

We just want the first 10 categories of the dataset, so that's why we create a new train dataset and test dataset with those labels as below

In [ ]:
#   Subset for the CIFAR100 in order to have just 10 categories
#Training Dataset
sub_X_train = X_train[np.isin(Y_train, [0,1,2,3,4,5,6,7,8,9]).flatten()]
sub_Y_train = Y_train[np.isin(Y_train, [0,1,2,3,4,5,6,7,8,9]).flatten()]

#Test Dataset
sub_X_test = X_test[np.isin(Y_test, [0,1,2,3,4,5,6,7,8,9]).flatten()]
sub_Y_test = Y_test[np.isin(Y_test, [0,1,2,3,4,5,6,7,8,9]).flatten()]

In [ ]:
print('Number of records for 10 classes X train',sub_X_train.shape)
print('Number of records for 10 classes X test',sub_X_test.shape)
print('Number of records for 10 classes Y train',sub_Y_train.shape)
print('Number of records for 10 classes Y test',sub_Y_test.shape)

In [ ]:
print("First 5 entries of X_train",sub_X_train[0:5])
print("First 5 entries of X_test",sub_X_test[0:5])
print("First 5 entries of Y_train",sub_Y_train[0:5])
print("First 5 entries of Y_test",sub_Y_test[0:5])

In [ ]:
plt.figure(figsize=(15,5))
for i in range(1,6):
  plt.subplot(1,5,i)
  plt.imshow(sub_X_train[i,:,:],cmap='gray')
  plt.title('Sign language of {}'.format(sub_Y_train[i]))
  plt.axis('off')
  plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression


For the Logistic Regression implementation, it's needed an X array with dimension <= 2. In this case, we have a dimension of 4, that's why we have to do a reshape as below

The same process has to be done for the other methods 

In [ ]:
sub_X_test_or=sub_X_test
sub_X_train_or=sub_X_train

sub_X_train_rshp=sub_X_train.reshape(5000, 32*32*3) ## Reshaping data
sub_X_test_rshp=sub_X_test.reshape(1000, 32*32*3)   ## Reshaping data  

In [ ]:
print(sub_X_train_rshp[0])

### Logistic Regression method

In [ ]:
lr_start_time=time.time()

In [ ]:
logreg =  LogisticRegression()
logreg.fit(sub_X_train_rshp,sub_Y_train) ## fitting the logistic regression model

In [ ]:
y_pred=logreg.predict(sub_X_test_rshp)
y_pred

In [ ]:
len(y_pred)

In [ ]:
len(sub_X_test)

In [ ]:
plt.figure(figsize=(15,5))
for i in range(1,6):
  plt.subplot(1,5,i)
  plt.imshow(sub_X_test[i,:,:],cmap='gray')
  plt.title('Sign language of {}'.format(Y_train[i]))
  plt.axis('off')
  plt.tight_layout()
plt.show()

#### Table for classification report and confusion

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.metrics import f1_score

In [ ]:
logreg.score(sub_X_test_rshp,sub_Y_test) ## Overall accuracy score of the logistic regression model

In [ ]:
lr_cm=confusion_matrix(y_pred,sub_Y_test) ## To obtain the confusion matrix
print(classification_report(y_pred,sub_Y_test))

print()
print("Execution Time %s seconds: " % (time.time() - lr_start_time))

In [ ]:
lr_cm

In [ ]:
f1_test = f1_score(y_pred, sub_Y_test, average='weighted')
print( "Test f1 score:", f1_test)

In [ ]:
y_pred

In [ ]:
sub_X_train

### Structure Predictions method

We procceed to create execute the Structure predictions method in order to identify differences and get some conclusions.

Here we implement the class IOKR2 due to some issues:

In [ ]:
# Implementation
import time
# from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel, rbf_kernel
# from sklearn.metrics import f1_score
import numpy as np
# import pandas as pd
# from stpredictiobns.datasets.load_data import load_bibtex
# from sklearn.model_selection import train_test_split
# import arff
from numpy.linalg import inv
# import os

# from line_profiler import LineProfiler

"""
Created on December 12, 2021
"""


class IOKR2:
    """
    Class used to apply Input and Output Kernel Regression
    """

    
    def __init__(self):
        """
        Initialization of the below parameters.
        Parameters
        ----------
        X_train :  sparse matrix - containing explanatory variables of the train set
        Y_train: sparse matrix - containing target variable of the train set
        Ky: output scalar kernel
        M: gram matrix on training set
        verbose: binary - display more parameters
        linear:
        """
        self.X_train = None
        self.Y_train = None
        self.Ky = None
        self.M = None
        self.verbose = 0
        self.input_kernel = None
        self.output_kernel = None

  
    def fit(self, X, Y, L, input_kernel='linear', input_kernel_param=None):
        """
        Model Fitting
        """

        # save input and output training data
        self.X_train, self.Y_train = X, Y

        # instantiate input kernel parameter when not given
        if input_kernel_param is None:
            if input_kernel == 'rbf':
                input_kernel_param = 1.
            elif input_kernel == 'polynomial':
                input_kernel_param = [3, None, 1]

        # define input kernel
        if input_kernel == 'linear':
            self.input_kernel = lambda A, B: linear_kernel(A, B)
        elif input_kernel == 'polynomial':
            self.input_kernel = lambda A, B: polynomial_kernel(A, B, degree=input_kernel_param[0],
                                                               gamma=input_kernel_param[1], coef0=input_kernel_param[2])
        elif input_kernel == 'rbf':
            self.input_kernel = lambda A, B: rbf_kernel(A, B, gamma=input_kernel_param)
        else:
            self.input_kernel = input_kernel

        # compute input gram matrix
        Kx = self.input_kernel(X, X)

        # kernel ridge regression training computation: n x n matrix inversion
        t0 = time.time()
        n = Kx.shape[0]
        self.M = np.linalg.inv(Kx + n * L * np.eye(n))
        if self.verbose > 0:
            print(f'Fitting time: {time.time() - t0} in s')

    def alpha(self, X_test):

        Kx = self.input_kernel(self.X_train, X_test)
        A = self.M.dot(Kx)

        return A


    def predict(self, X_test, Y_candidates, output_kernel='linear', output_kernel_param=None):

        """
        Model Prediction
        """

        # instantiate output kernel parameter when not given
        if output_kernel_param is None:
            if output_kernel == 'rbf':
                output_kernel_param = 1.
            elif output_kernel == 'polynomial':
                output_kernel_param = [3, None, 1]

        # define output kernel
        if output_kernel == 'linear':
            self.output_kernel = lambda A, B: linear_kernel(A, B)
        elif output_kernel == 'polynomial':
            self.output_kernel = lambda A, B: polynomial_kernel(A, B, degree=output_kernel_param[0],
                                                                gamma=output_kernel_param[1],
                                                                coef0=output_kernel_param[2])
        elif output_kernel == 'rbf':
            self.output_kernel = lambda A, B: rbf_kernel(A, B, gamma=output_kernel_param)
        else:
            self.output_kernel = output_kernel

        # compute output gram matrix
        Ky = self.output_kernel(self.Y_train, Y_candidates)

        # compute prediction
        t0 = time.time()
        Alpha = self.alpha(X_test)
        scores = Ky.transpose().dot(Alpha)
        idx_pred = np.argmax(scores, axis=0)
        Y_pred = Y_candidates[idx_pred]
        if self.verbose > 0:
            print(f'Decoding time: {time.time() - t0} in s')

        return Ky,Alpha,scores,idx_pred,Y_pred


In [ ]:
from stpredictions.models.IOKR.model import IOKR


In [ ]:
nb_class=10

In [ ]:
sp_start_time= time.time()

In [ ]:
print(sub_X_train_rshp.shape)
print(sub_X_test_rshp.shape)
print(sub_Y_train.shape)
print(sub_Y_test.shape)

In [ ]:
Y_tra=np.zeros([len(sub_Y_train),nb_class])
Y_tes=np.zeros([len(sub_Y_test),nb_class])
for t in range(len(sub_Y_train)):
  Y_tra[t][sub_Y_train[t][0]]=1
for t in range(len(sub_Y_test)):
  Y_tes[t][sub_Y_test[t][0]]=1

In [ ]:
clf = IOKR2()
clf.verbose = 1
L =1e-6
clf.fit(X=sub_X_train_rshp, Y=Y_tra, L=L, input_kernel='linear')
Ky,Alpha,scores,idx_pred,Y_pred_test = clf.predict(X_test=sub_X_test_rshp, Y_candidates=np.eye(10))
#f1_test_st = f1_score(Y_pred_test, Y_test, average='weighted')
#print( "Test f1 score:", f1_test_st)
print(Y_pred_test[1])

In [ ]:
Y_pred_test

In [ ]:
y_res_fin=np.argmax(Y_tes,axis=1)

In [ ]:
y_test_fin=np.argmax(Y_pred_test,axis=1)

In [ ]:
sub_X_train

In [ ]:
idx_pred

#### Table for classification report and confusion

In [ ]:
sp_cm=confusion_matrix(idx_pred,y_res_fin) ## To obtain the confusion matrix
print(classification_report(idx_pred,y_res_fin))

print()
print("Execution Time %s seconds: " % (time.time() - sp_start_time))

### Random Forest method

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf_start_time = time.time()

In [ ]:
sub_X_train = sub_X_train.astype('float32')
sub_X_test = sub_X_test.astype('float32')

In [ ]:
sub_X_train

In [ ]:
print(sub_X_train.shape[0], 'train samples')
print(sub_X_test.shape[0], 'test samples')

In [ ]:
# fit a RandomForest model to the data
model = RandomForestClassifier(n_estimators = 10)

In [ ]:
model.fit(sub_X_train_rshp, sub_Y_train)
print(model)

In [ ]:
cv_results = cross_val_score(model, sub_X_train_rshp, sub_Y_train,
                cv = 2, scoring='accuracy', n_jobs = -1, verbose = 1)

#model.fit(sub_X_train, sub_Y_train)

print(); print(cv_results)
#print(); print(model)

In [ ]:
# make predictions
expected_y  = sub_Y_test
predicted_y = model.predict(sub_X_test_rshp)

In [ ]:
sub_X_test_rshp

In [ ]:
sub_X_test_rshp.shape

In [ ]:
print(predicted_y[0:3])

#### Table for classification report and confusion

In [ ]:
# summarize the fit of the model
print(); print(classification_report(expected_y, predicted_y))
#print(); print(metrics.confusion_matrix(expected_y, predicted_y))

print()
print("Execution Time %s seconds: " % (time.time() - rf_start_time))   

In [ ]:
rf_cm=confusion_matrix(predicted_y,sub_Y_test) ## To obtain the confusion matrix
rf_cm

In [ ]:
predicted_y